# 🎬 Исследование Рекомендательных Систем и ML Аналитика
**Дисциплина:** Машинное Обучение  
**Задачи:**
1. Анализ распределения данных и расчет разреженности матрицы (Sparsity);
2. Расчет популярности фильмов по байесовской формуле IMDb;
3. Построение коллаборативной фильтрации (User-Based и Item-Based);
4. Построение контентной фильтрации (TF-IDF + Cosine Similarity);
5. Факторизация матрицы методом Truncated SVD и анализ скрытых факторов (Scree Plot);
6. 2D PCA проекция семантического пространства фильмов;
7. Сравнение качества моделей по метрикам RMSE, MAE, Precision@K.

In [ ]:
import sys
import os
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer

from database.connection import SessionLocal
from database.models import Movie, Rating, User
from ml_engine.evaluation import MLEvaluationEngine

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline
print('Библиотеки успешно загружены!')

## 1. Загрузка данных из БД MySQL и расчет разреженности матрицы

In [ ]:
db = SessionLocal()
eval_engine = MLEvaluationEngine()
summary = eval_engine.get_dataset_summary(db)

print('Сводка по датасету:')
for k, v in summary.items():
    print(f'  {k}: {v}')

## 2. Распределение оценок (Rating Distribution & KDE)

In [ ]:
ratings = db.query(Rating).all()
df_ratings = pd.DataFrame([{'user_id': r.user_id, 'movie_id': r.movie_id, 'rating': float(r.rating)} for r in ratings])

plt.figure(figsize=(9, 4))
sns.histplot(df_ratings['rating'], bins=9, kde=True, color='teal', edgecolor='black')
plt.title('Распределение пользовательских оценок (1.0 - 5.0 ⭐)', fontsize=13)
plt.xlabel('Оценка')
plt.ylabel('Частота')
plt.axvline(df_ratings['rating'].mean(), color='red', linestyle='--', label=f'Среднее: {df_ratings["rating"].mean():.2f}')
plt.legend()
plt.show()

## 3. Матрица взаимодействий и тепловая карта косинусного сходства

In [ ]:
sim_sample = eval_engine.get_similarity_heatmap_sample(db, limit=8)
titles = sim_sample['titles']
matrix = np.array(sim_sample['matrix'])

plt.figure(figsize=(8, 6))
sns.heatmap(matrix, annot=True, fmt='.2f', cmap='coolwarm', xticklabels=titles, yticklabels=titles)
plt.title('Тепловая карта сходства популярных фильмов (Cosine Similarity)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 4. Scree Plot: Латентные факторы SVD (Метод локтя)

In [ ]:
scree_data = eval_engine.get_svd_scree_data(db)

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.bar(scree_data['components'], scree_data['singular_values'], color='royalblue', alpha=0.8, label='Сингулярные значения')
ax1.set_ylabel('Singular Value', color='royalblue')

ax2 = ax1.twinx()
ax2.plot(scree_data['components'], scree_data['cumulative_variance_pct'], color='crimson', marker='o', linewidth=2, label='Кумулятивная дисперсия (%)')
ax2.set_ylabel('Накопленная дисперсия (%)', color='crimson')
ax2.set_ylim(0, 105)

plt.title('Scree Plot: Анализ латентных компонентов матрицы рейтингов (SVD)', fontsize=12)
plt.show()

## 5. 2D PCA Проекция пространства признаков фильмов (TF-IDF)

In [ ]:
pca_data = eval_engine.get_pca_movie_projection(db)
df_pca = pd.DataFrame(pca_data)

plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_pca, x='pc1', y='pc2', hue='genre', s=120, palette='tab10')

for _, row in df_pca.iterrows():
    plt.annotate(row['title'], (row['pc1']+0.01, row['pc2']+0.01), fontsize=8, alpha=0.85)

plt.title('2D PCA Проекция пространства метаданных фильмов', fontsize=13)
plt.xlabel('Главная компонента 1 (PC1)')
plt.ylabel('Главная компонента 2 (PC2)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 6. Сравнение качества моделей: RMSE, MAE, Precision, Recall

In [ ]:
metrics_res = eval_engine.evaluate_models(db)
df_metrics = pd.DataFrame(metrics_res['models']).T
display(df_metrics)

df_metrics[['rmse', 'mae']].plot(kind='bar', figsize=(9, 4), colormap='Set2')
plt.title('Сравнение ошибок прогноза моделей (RMSE vs MAE)', fontsize=12)
plt.ylabel('Ошибка')
plt.xticks(rotation=15)
plt.show()

db.close()